# BRRI Win2024 — Google Drive ফোল্ডার তৈরি (Photo Folders)

এই notebook **ডেটা সংগ্রহ ফর্ম**-এর ছবি টেবিল অনুযায়ী Google Drive-এ ফোল্ডার বানায়।

**কাজ:**
1. Google-এ লগইন
2. `BRRI_Win2024_Field_Photos` (মূল ফোল্ডার) + ২০টি উপ-ফোল্ডার তৈরি
3. প্রতিটি ফোল্ডারের লিংক প্রিন্ট — ফর্মের **Google Drive লিংক** কলামে বসান

**অতিরিক্ত ছবি:** এক ফোল্ডারে একাধিক ফото OK — `10_b65_marking.jpg`, `10_belt_worn.jpg` (NN_ prefix রাখুন)। ফর্মে extra row লাগে না।

**চালানো:** Runtime → Run all

In [ ]:
# Install Drive API client (Colab)
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from __future__ import annotations

import re
from datetime import date

from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive = build("drive", "v3")
print("Google Drive authenticated.")

In [ ]:
# --- Config (matches docs/generate_data_collection_form.py PHOTO_SLOTS) ---

ROOT_FOLDER_NAME = "BRRI_Win2024_Field_Photos"  # add district: f"BRRI_Win2024_Field_Photos_Patuakhali"
DISTRICT_SUFFIX = ""  # e.g. "Patuakhali" → BRRI_Win2024_Field_Photos_Patuakhali

if DISTRICT_SUFFIX.strip():
    ROOT_FOLDER_NAME = f"{ROOT_FOLDER_NAME}_{DISTRICT_SUFFIX.strip()}"

# (photo_no, folder_name, suggested_filename, part_paper)
PHOTO_FOLDERS = [
    ("01", "01_MAIN_FRAME", "01_main_frame.jpg", "MAIN FRAME"),
    ("02", "02_HOPPER", "02_hopper.jpg", "HOPPER"),
    ("03", "03_AIR_CONTROL_PLATE", "03_air_control_plate.jpg", "AIR CONTROL PLATE"),
    ("04", "04_GRAIN_CONTROL_PLATE", "04_grain_control_plate.jpg", "GRAIN CONTROL PLATE"),
    ("05", "05_BLOWER_UNITE", "05_blower_unite.jpg", "BLOWER UNITE"),
    ("06", "06_BLOWER_COVER", "06_blower_cover.jpg", "BLOWER COVER PLATE"),
    ("07", "07_AIR_OUTLET_CONTROL", "07_air_outlet_control.jpg", "AIR OUTLET CONTROL PLATE"),
    ("08", "08_SIEVE", "08_sieve.jpg", "SIEVE (THREE TYPE)"),
    ("09", "09_SIEVE_SHAFT", "09_sieve_shaft.jpg", "SIEVE SHAFT"),
    ("10", "10_POWER_PULLEY_BELT", "10_b65_marking.jpg", "POWER PULLEY BELT (B65)"),
    ("11", "11_MOTOR", "11_motor.jpg", "MOTOR"),
    ("12", "12_MOTOR_PULLEY", "12_motor_pulley.jpg", "MOTOR PULLEY"),
    ("13", "13_BLOWER_PULLEY", "13_blower_pulley.jpg", "BLOWER PULLEY"),
    ("14", "14_PILLOW_BEARING_UCP206", "14_pillow_bearing.jpg", "PILLOW BEARING UCP206"),
    ("15", "15_BALL_BEARING_6302", "15_ball_bearing.jpg", "BALL BEARING-6302"),
    ("16", "16_BEARING_6203", "16_bearing_6203.jpg", "BEARING 6203 (sieve)"),
    ("17", "17_GRAIN_OUTLET", "17_grain_outlet.jpg", "GRAIN OUTLET"),
    ("18", "18_WINNOWER_SHOW_COVER", "18_show_cover.jpg", "WINNOWER SHOW COVER"),
    ("19", "19_FULL_MACHINE_FRONT", "19_full_front.jpg", "Full machine — front"),
    ("20", "20_FULL_MACHINE_SIDE", "20_full_side.jpg", "Full machine — side"),
]

print(f"Root folder: {ROOT_FOLDER_NAME}")
print(f"Subfolders: {len(PHOTO_FOLDERS)}")

In [ ]:
def folder_link(folder_id: str) -> str:
    return f"https://drive.google.com/drive/folders/{folder_id}"


def find_folder(name: str, parent_id: str | None = None) -> str | None:
    """Return folder id if it exists under parent (or My Drive root)."""
    q = (
        f"name = '{name.replace(chr(39), chr(92)+chr(39))}' "
        "and mimeType = 'application/vnd.google-apps.folder' "
        "and trashed = false"
    )
    if parent_id:
        q += f" and '{parent_id}' in parents"
    resp = drive.files().list(q=q, spaces="drive", fields="files(id, name)").execute()
    files = resp.get("files", [])
    return files[0]["id"] if files else None


def create_folder(name: str, parent_id: str | None = None) -> str:
    """Create folder or return existing id (safe to re-run)."""
    existing = find_folder(name, parent_id)
    if existing:
        return existing
    meta = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
    }
    if parent_id:
        meta["parents"] = [parent_id]
    created = drive.files().create(body=meta, fields="id").execute()
    return created["id"]


def write_readme(folder_id: str, lines: list[str]) -> None:
    """Optional: upload a small README.txt so collectors know expected filename."""
    content = "\n".join(lines).encode("utf-8")
    from googleapiclient.http import MediaInMemoryUpload

    name = "README_upload_here.txt"
    q = f"name = '{name}' and '{folder_id}' in parents and trashed = false"
    existing = drive.files().list(q=q, fields="files(id)").execute().get("files", [])
    media = MediaInMemoryUpload(content, mimetype="text/plain")
    if existing:
        drive.files().update(fileId=existing[0]["id"], media_body=media).execute()
    else:
        drive.files().create(
            body={"name": name, "parents": [folder_id]},
            media_body=media,
            fields="id",
        ).execute()

In [ ]:
CREATE_README = True  # small hint file in each subfolder

root_id = create_folder(ROOT_FOLDER_NAME)
root_url = folder_link(root_id)

rows = []
for photo_no, folder_name, filename, part in PHOTO_FOLDERS:
    sub_id = create_folder(folder_name, parent_id=root_id)
    sub_url = folder_link(sub_id)
    if CREATE_README:
        write_readme(
            sub_id,
            [
                f"BRRI Win2024 — Photo #{photo_no}",
                f"Part (paper): {part}",
                f"Upload filename: {filename}",
                f"Created: {date.today().isoformat()}",
                "",
                "Paste this folder link in the data collection form.",
            ],
        )
    rows.append(
        {
            "photo_no": photo_no,
            "folder_name": folder_name,
            "filename": filename,
            "part_paper": part,
            "folder_link": sub_url,
        }
    )

print("=" * 60)
print("ROOT FOLDER (paste in form → meta.drive_root_link)")
print(root_url)
print("=" * 60)
print("\nFULL FOLDER LINKS (copy to docs/photo_folder_links.csv):")
for r in rows:
    print(f"{r['photo_no']}\t{r['folder_link']}")

In [ ]:
import pandas as pd

csv_rows = [
    {
        "photo_no": "root",
        "folder_name": ROOT_FOLDER_NAME,
        "filename": "",
        "part_paper": "",
        "folder_link": root_url,
    },
    *rows,
]
df = pd.DataFrame(csv_rows)
display(df[["photo_no", "folder_name", "filename", "folder_link"]])

csv_name = "photo_folder_links.csv"
df.to_csv(csv_name, index=False)
from google.colab import files

files.download(csv_name)
print(
    "Downloaded photo_folder_links.csv — copy to repo docs/ then run:\n"
    "  backend/.venv/bin/python docs/generate_data_collection_form.py"
)

## ফর্মে বসানোর ক্রম

1. **সংগ্রহকারীর তথ্য** → `Google Drive মূল ফোল্ডার লিংক` = ROOT link উপরে
2. **ছবি সংগ্রহ** → প্রতি সারির `Google Drive লিংক` = CSV/table থেকে matching `folder_link`
3. মোবাইল থেকে ছবি তুলে সরাসরি সেই subfolder-এ upload করুন
4. **অতিরিক্ত ছবি:** same folder, name like `10_belt_worn.jpg` — no extra form row

Notebook আবার চালালে একই নামের ফোল্ডার থাকলে **নতুন বানাবে না** (duplicate safe)।